# Notebook 02 — Environment

## Projet : C-MARL pour la réduction des absences aux rendez-vous médicaux

Ce notebook construit l'environnement d'apprentissage par renforcement à partir des fichiers préparés dans le Notebook 01.

Objectifs :
- charger `easy_data.csv`, `medium_data.csv` et `hard_data.csv` ;
- définir les actions possibles pour chaque stage ;
- définir l'état observé par les agents ;
- définir la fonction de récompense ;
- tester l'environnement pour Easy, Medium et Hard.


## Cellule 1 — Imports

In [1]:
import os
import numpy as np
import pandas as pd

## Cellule 2 — Chemins du projet

Si ce notebook est placé dans `medical_c_marl_project/notebooks/`, ce code retourne automatiquement au dossier principal du projet.

In [2]:
PROJECT_PATH = os.path.abspath("..")
RESULTS_DIR = os.path.join(PROJECT_PATH, "results")

EASY_PATH = os.path.join(RESULTS_DIR, "easy_data.csv")
MEDIUM_PATH = os.path.join(RESULTS_DIR, "medium_data.csv")
HARD_PATH = os.path.join(RESULTS_DIR, "hard_data.csv")

print("Project path:", PROJECT_PATH)
print("Results path:", RESULTS_DIR)
print("Easy path:", EASY_PATH)
print("Medium path:", MEDIUM_PATH)
print("Hard path:", HARD_PATH)

Project path: C:\Users\UltraPc\Documents\ML PROBA\C-marl
Results path: C:\Users\UltraPc\Documents\ML PROBA\C-marl\results
Easy path: C:\Users\UltraPc\Documents\ML PROBA\C-marl\results\easy_data.csv
Medium path: C:\Users\UltraPc\Documents\ML PROBA\C-marl\results\medium_data.csv
Hard path: C:\Users\UltraPc\Documents\ML PROBA\C-marl\results\hard_data.csv


## Cellule 3 — Vérifier que les fichiers existent

Ces fichiers doivent avoir été créés par le Notebook 01.

In [3]:
for path in [EASY_PATH, MEDIUM_PATH, HARD_PATH]:
    if os.path.exists(path):
        print("Fichier trouvé:", path)
    else:
        print("Fichier introuvable:", path)

Fichier trouvé: C:\Users\UltraPc\Documents\ML PROBA\C-marl\results\easy_data.csv
Fichier trouvé: C:\Users\UltraPc\Documents\ML PROBA\C-marl\results\medium_data.csv
Fichier trouvé: C:\Users\UltraPc\Documents\ML PROBA\C-marl\results\hard_data.csv


## Cellule 4 — Charger les trois datasets

In [4]:
easy_data = pd.read_csv(EASY_PATH)
medium_data = pd.read_csv(MEDIUM_PATH)
hard_data = pd.read_csv(HARD_PATH)

print("Easy data:", easy_data.shape)
print("Medium data:", medium_data.shape)
print("Hard data:", hard_data.shape)

Easy data: (110521, 4)
Medium data: (110521, 10)
Hard data: (110521, 13)


## Cellule 5 — Aperçu des données

On vérifie que `No_show` est bien présente, car elle servira uniquement au calcul de la récompense. Elle ne sera pas donnée directement comme état à l'agent.

In [5]:
display(easy_data.head())
display(medium_data.head())
display(hard_data.head())

,AgeGroup,Gender,SMS_received,No_show
0,3,0,0,0
1,2,1,0,0
2,3,0,0,0
3,0,0,0,0
4,2,0,0,0


,AgeGroup,Gender,SMS_received,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,MedicalRisk,No_show
0,3,0,0,0,1,0,0,0,1,0
1,2,1,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,2,0,0,0,1,1,0,0,1,0


,AgeGroup,Gender,SMS_received,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,MedicalRisk,WaitingGroup,AppointmentWeekday,Neighbourhood,No_show
0,3,0,0,0,1,0,0,0,1,0,4,39,0
1,2,1,0,0,0,0,0,0,0,0,4,39,0
2,3,0,0,0,0,0,0,0,0,0,4,45,0
3,0,0,0,0,0,0,0,0,0,0,4,54,0
4,2,0,0,0,1,1,0,0,1,0,4,39,0


## Cellule 6 — Définir l'environnement RL

La classe suivante transforme un dataset en environnement d'apprentissage par renforcement.

Logique :
- une ligne = un rendez-vous médical ;
- un état = les informations du patient sans `No_show` ;
- une action = décision du système ;
- une récompense = qualité de la décision ;
- un épisode = passage sur un nombre limité de patients.

Pour garder l'entraînement rapide, on utilise `max_steps`. Cela évite de parcourir les 110 000 lignes à chaque épisode.

In [6]:
class MedicalAppointmentEnv:
    """
    Environnement simple pour le projet C-MARL.

    Chaque ligne du dataset représente un rendez-vous médical.
    L'environnement retourne un état, reçoit une action et calcule une récompense.
    """

    def __init__(self, data, stage, max_steps=1000, random_start=True):
        self.data = data.reset_index(drop=True)
        self.stage = stage.lower()
        self.max_steps = max_steps
        self.random_start = random_start

        self.current_index = 0
        self.steps = 0

        if self.stage == "easy":
            self.actions = [0, 1]
            self.action_names = {
                0: "ne rien faire",
                1: "envoyer rappel"
            }

        elif self.stage == "medium":
            self.actions = [0, 1, 2]
            self.action_names = {
                0: "ne rien faire",
                1: "envoyer rappel",
                2: "prioriser"
            }

        elif self.stage == "hard":
            self.actions = [0, 1, 2, 3]
            self.action_names = {
                0: "ne rien faire",
                1: "envoyer rappel",
                2: "prioriser",
                3: "proposer reprogrammation"
            }

        else:
            raise ValueError("stage must be: easy, medium, or hard")

        self.n_actions = len(self.actions)

    def reset(self):
        """
        Réinitialise l'environnement au début d'un nouvel épisode.
        """
        self.steps = 0

        if self.random_start:
            max_start = max(1, len(self.data) - self.max_steps)
            self.current_index = np.random.randint(0, max_start)
        else:
            self.current_index = 0

        return self._get_state()

    def _get_state(self):
        """
        Retourne l'état actuel.
        L'état contient les variables du patient, sans la colonne No_show.
        """
        if self.current_index >= len(self.data):
            return None

        patient = self.data.iloc[self.current_index]
        state = patient.drop("No_show").values

        return tuple(state.astype(int))

    def step(self, action):
        """
        Exécute une action, calcule la récompense et passe au patient suivant.
        """
        if action not in self.actions:
            raise ValueError(f"Action invalide: {action}. Actions possibles: {self.actions}")

        patient = self.data.iloc[self.current_index]
        reward = self._calculate_reward(patient, action)

        self.current_index += 1
        self.steps += 1

        done = self.steps >= self.max_steps or self.current_index >= len(self.data)
        next_state = None if done else self._get_state()

        return next_state, reward, done

    def _calculate_reward(self, patient, action):
        """
        Calcule la récompense commune des agents.

        No_show = 0 : le patient est venu.
        No_show = 1 : le patient n'est pas venu.
        """
        no_show = int(patient["No_show"])
        reward = 0

        if self.stage == "easy":
            if no_show == 1 and action == 1:
                reward += 2
            elif no_show == 1 and action == 0:
                reward -= 2
            elif no_show == 0 and action == 0:
                reward += 1
            elif no_show == 0 and action == 1:
                reward -= 0.5

        elif self.stage == "medium":
            medical_risk = int(patient["MedicalRisk"])

            if no_show == 1 and action == 1:
                reward += 3
            elif medical_risk == 1 and action == 2:
                reward += 2
            elif no_show == 1 and action == 0:
                reward -= 3
            elif no_show == 0 and action == 0:
                reward += 1
            else:
                reward -= 1

        elif self.stage == "hard":
            medical_risk = int(patient["MedicalRisk"])
            waiting_group = int(patient["WaitingGroup"])

            if no_show == 1 and action in [1, 3]:
                reward += 5
            elif no_show == 1 and action == 0:
                reward -= 5
            elif medical_risk == 1 and action == 2:
                reward += 3
            elif waiting_group >= 2 and action == 3:
                reward += 3
            elif no_show == 0 and action == 0:
                reward += 2
            else:
                reward -= 2

        return reward

    def get_action_space(self):
        return self.n_actions

    def get_action_names(self):
        return self.action_names

    def get_number_of_patients(self):
        return len(self.data)


## Cellule 7 — Tester l'environnement Easy

Dans Easy, les actions possibles sont :
- `0` : ne rien faire ;
- `1` : envoyer un rappel.

In [7]:
env_easy = MedicalAppointmentEnv(easy_data, stage="easy", max_steps=10, random_start=False)

state = env_easy.reset()

print("Premier état Easy:", state)
print("Nombre d'actions:", env_easy.get_action_space())
print("Actions:", env_easy.get_action_names())
print("Nombre de patients:", env_easy.get_number_of_patients())

next_state, reward, done = env_easy.step(action=1)

print("Récompense après action 1:", reward)
print("État suivant:", next_state)
print("Fin de l'épisode ?", done)

Premier état Easy: (np.int64(3), np.int64(0), np.int64(0))
Nombre d'actions: 2
Actions: {0: 'ne rien faire', 1: 'envoyer rappel'}
Nombre de patients: 110521
Récompense après action 1: -0.5
État suivant: (np.int64(2), np.int64(1), np.int64(0))
Fin de l'épisode ? False


## Cellule 8 — Tester l'environnement Medium

Dans Medium, on ajoute l'action `2 = prioriser`.

In [8]:
env_medium = MedicalAppointmentEnv(medium_data, stage="medium", max_steps=10, random_start=False)

state = env_medium.reset()

print("Premier état Medium:", state)
print("Nombre d'actions:", env_medium.get_action_space())
print("Actions:", env_medium.get_action_names())

next_state, reward, done = env_medium.step(action=2)

print("Récompense après action 2:", reward)
print("État suivant:", next_state)
print("Fin de l'épisode ?", done)

Premier état Medium: (np.int64(3), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1))
Nombre d'actions: 3
Actions: {0: 'ne rien faire', 1: 'envoyer rappel', 2: 'prioriser'}
Récompense après action 2: 2
État suivant: (np.int64(2), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0))
Fin de l'épisode ? False


## Cellule 9 — Tester l'environnement Hard

Dans Hard, on ajoute l'action `3 = proposer reprogrammation`.

In [9]:
env_hard = MedicalAppointmentEnv(hard_data, stage="hard", max_steps=10, random_start=False)

state = env_hard.reset()

print("Premier état Hard:", state)
print("Nombre d'actions:", env_hard.get_action_space())
print("Actions:", env_hard.get_action_names())

next_state, reward, done = env_hard.step(action=3)

print("Récompense après action 3:", reward)
print("État suivant:", next_state)
print("Fin de l'épisode ?", done)

Premier état Hard: (np.int64(3), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(4), np.int64(39))
Nombre d'actions: 4
Actions: {0: 'ne rien faire', 1: 'envoyer rappel', 2: 'prioriser', 3: 'proposer reprogrammation'}
Récompense après action 3: -2
État suivant: (np.int64(2), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(4), np.int64(39))
Fin de l'épisode ? False


## Cellule 10 — Simulation aléatoire sur un épisode

Cette cellule vérifie que l'environnement fonctionne sur plusieurs patients. Les actions sont choisies au hasard. Ce n'est pas encore l'apprentissage ; l'apprentissage sera fait dans le Notebook 03.

In [10]:
def run_random_episode(env):
    state = env.reset()
    total_reward = 0
    done = False
    decisions = []

    while not done:
        action = np.random.choice(env.actions)
        next_state, reward, done = env.step(action)
        total_reward += reward

        decisions.append({
            "action": action,
            "action_name": env.action_names[action],
            "reward": reward
        })

        state = next_state

    return total_reward, pd.DataFrame(decisions)

random_reward_easy, random_decisions_easy = run_random_episode(
    MedicalAppointmentEnv(easy_data, stage="easy", max_steps=20, random_start=True)
)

print("Reward totale Easy avec décisions aléatoires:", random_reward_easy)
random_decisions_easy.head(10)

Reward totale Easy avec décisions aléatoires: 2.0


,action,action_name,reward
0,1,envoyer rappel,-0.5
1,1,envoyer rappel,-0.5
2,1,envoyer rappel,-0.5
3,0,ne rien faire,1.0
4,1,envoyer rappel,-0.5
5,0,ne rien faire,1.0
6,1,envoyer rappel,-0.5
7,0,ne rien faire,1.0
8,0,ne rien faire,1.0
9,1,envoyer rappel,-0.5


## Cellule 11 — Comparer une simulation aléatoire sur les 3 stages

In [11]:
random_results = []

for stage_name, stage_data in [
    ("easy", easy_data),
    ("medium", medium_data),
    ("hard", hard_data)
]:
    env = MedicalAppointmentEnv(stage_data, stage=stage_name, max_steps=200, random_start=True)
    total_reward, _ = run_random_episode(env)

    random_results.append({
        "Stage": stage_name,
        "Random total reward": total_reward,
        "Number of actions": env.get_action_space()
    })

random_results_df = pd.DataFrame(random_results)
random_results_df

,Stage,Random total reward,Number of actions
0,easy,56.0,2
1,medium,117.0,3
2,hard,-49.0,4


## Cellule 12 — Sauvegarder un résumé de l'environnement

On sauvegarde un petit fichier CSV qui résume les actions disponibles dans chaque stage. Cela servira pour le rapport.

In [12]:
summary_rows = []

for stage_name, stage_data in [
    ("easy", easy_data),
    ("medium", medium_data),
    ("hard", hard_data)
]:
    env = MedicalAppointmentEnv(stage_data, stage=stage_name, max_steps=10)
    summary_rows.append({
        "Stage": stage_name,
        "Number of patients": env.get_number_of_patients(),
        "Number of state variables": stage_data.shape[1] - 1,
        "Number of actions": env.get_action_space(),
        "Actions": str(env.get_action_names())
    })

environment_summary = pd.DataFrame(summary_rows)

environment_summary_path = os.path.join(RESULTS_DIR, "environment_summary.csv")
environment_summary.to_csv(environment_summary_path, index=False)

print("Résumé sauvegardé dans:", environment_summary_path)
environment_summary

Résumé sauvegardé dans: C:\Users\UltraPc\Documents\ML PROBA\C-marl\results\environment_summary.csv


,Stage,Number of patients,Number of state variables,Number of actions,Actions
0,easy,110521,3,2,"{0: 'ne rien faire', 1: 'envoyer rappel'}"
1,medium,110521,9,3,"{0: 'ne rien faire', 1: 'envoyer rappel', 2: '..."
2,hard,110521,12,4,"{0: 'ne rien faire', 1: 'envoyer rappel', 2: '..."


## Cellule 13 — Message final

Si toutes les cellules fonctionnent, l'environnement est prêt pour l'entraînement Q-learning dans le Notebook 03.

In [13]:
print("Notebook 02 terminé avec succès.")
print("Prochaine étape : Notebook 03 — Agents et entraînement Q-learning avec checkpoints.")

Notebook 02 terminé avec succès.
Prochaine étape : Notebook 03 — Agents et entraînement Q-learning avec checkpoints.
